# 04 Lakeflow Declarative Pipelines

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Build a pipeline the declarative way with <b>Lakeflow Declarative Pipelines</b> (formerly Delta Live Tables, DLT): define <b>streaming tables</b> and <b>materialized views</b> in SQL, add <b>expectations</b>, create and run the pipeline, process new files incrementally, and read the pipeline's <b>event log</b>. Compare it with the same pipeline written by hand.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Declarative pipelines take over the plumbing you'd otherwise write yourself: dependency order, incremental processing, checkpoints, retries, data quality metrics and table maintenance. They're a major part of the exam's ingestion, transformation and jobs domains, where you're expected to know streaming tables vs materialized views, expectations, and pipeline modes.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Building blocks</b><br>
| Object | What it is | Typical layer |<br>|---|---|---|<br>| <b>Streaming table</b> | A Delta table fed by a stream: each record is processed once, incrementally (append-only sources, Auto Loader, Kafka) | Bronze, Silver |<br>| <b>Materialized view</b> | A Delta table holding the result of a query, recomputed (incrementally when possible) on each update | Silver, Gold |<br>| <b>View</b> | Temporary logic inside the pipeline, not published | Intermediate steps |<br>| <b>Flow</b> | How data moves into a target: append flows, AUTO CDC flows (formerly <code>APPLY CHANGES</code>) | Any |<br>| <b>Expectation</b> | A data quality rule: warn (keep), <code>DROP ROW</code>, or <code>FAIL UPDATE</code> | Silver |<br><br>
Pipeline settings: target <b>catalog and schema</b>, <b>serverless</b> compute, <b>triggered</b> (run, then stop) or <b>continuous</b>, <b>development</b> mode (keeps compute warm, no retries) or <b>production</b> mode. A <b>full refresh</b> reprocesses everything from scratch.<br><br>
You declare <b>what</b> each table should contain. The pipeline works out <b>how</b>: order, incremental state, retries.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A team maintained 2,000 lines of PySpark for a Bronze → Silver → Gold pipeline: file tracking, checkpoint paths, <code>MERGE</code> logic, ordering between 14 tables, and a homemade data quality log. Rewritten as a declarative pipeline, it became about 300 lines of SQL. Dependencies are inferred from the queries, expectations produce quality metrics automatically, and a new table is one more <code>CREATE OR REFRESH</code> statement.
</div>

## Syntax

```sql
CREATE OR REFRESH STREAMING TABLE orders_bronze
AS SELECT *, _metadata.file_name AS source_file
FROM STREAM read_files('/Volumes/workspace/workflows/files/ldp_orders/', format => 'csv', header => true);

CREATE OR REFRESH STREAMING TABLE orders_silver (
  CONSTRAINT valid_amount EXPECT (amount > 0) ON VIOLATION DROP ROW,
  CONSTRAINT has_customer EXPECT (customer_id IS NOT NULL)                -- warn only
) AS SELECT order_id, customer_id, amount FROM STREAM(orders_bronze);

CREATE OR REFRESH MATERIALIZED VIEW revenue_by_customer
AS SELECT customer_id, SUM(amount) AS revenue FROM orders_silver GROUP BY customer_id;
```

```python
from pyspark import pipelines as dp          # older code: import dlt

@dp.table                                     # streaming table when the function returns a stream
@dp.expect_or_drop("valid_amount", "amount > 0")
def orders_silver():
    return spark.readStream.table("orders_bronze")
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the schema and a volume folder, and writes the first batch of raw order files: 6 orders, including one with a negative amount and one without a customer.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The path of the first raw file.
</div>

In [0]:
CATALOG = "workspace"
DB = f"{CATALOG}.workflows"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {DB}.files")
BASE = f"/Volumes/{CATALOG}/workflows/files"

In [0]:
import os, shutil, time
RAW = f"{BASE}/ldp_orders"
shutil.rmtree(RAW, ignore_errors=True)
os.makedirs(RAW, exist_ok=True)

def write_batch(name, rows):
    with open(f"{RAW}/{name}.csv", "w") as f:
        f.write("order_id,customer_id,amount,order_ts\n" + "\n".join(rows) + "\n")
    return f"{RAW}/{name}.csv"

print(write_batch("batch_1", ["1,101,25.0,2024-11-01 09:00:00", "2,102,40.0,2024-11-01 10:30:00",
                              "3,101,-5.0,2024-11-01 11:00:00", "4,,12.5,2024-11-01 12:00:00",
                              "5,103,60.0,2024-11-02 08:15:00", "6,102,15.0,2024-11-02 09:45:00"]))

## 1. The hand-written version

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds the pipeline imperatively, the way you would without declarative pipelines: read the files, keep track of which files were already loaded, append new ones to Bronze, filter into Silver, recompute Gold.<br><br>
<b>Why it matters</b><br>Writing it once by hand shows what a declarative pipeline automates: the file bookkeeping, the ordering, the quality filtering and counting. In this version all of that is your code to maintain.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Bronze has 6 rows, Silver 4 (the negative amount and the missing customer are removed), and Gold has revenue for 3 customers.
</div>

In [0]:
from pyspark.sql import functions as F

SCHEMA = "order_id INT, customer_id INT, amount DOUBLE, order_ts TIMESTAMP"
processed_files = set()                                    # you must persist this somewhere in real life

def manual_update():
    files = sorted(f for f in os.listdir(RAW) if f.endswith(".csv") and f not in processed_files)
    if files:
        new = (spark.read.option("header", True).schema(SCHEMA).csv([f"{RAW}/{f}" for f in files])
               .withColumn("source_file", F.col("_metadata.file_name")))
        new.write.mode("append").saveAsTable(f"{DB}.manual_orders_bronze")
        processed_files.update(files)
    bronze = spark.table(f"{DB}.manual_orders_bronze")
    silver = bronze.filter("amount > 0 AND customer_id IS NOT NULL").select("order_id", "customer_id", "amount", "order_ts")
    print(f"  dropped by quality rules: {bronze.count() - silver.count()}")
    silver.write.mode("overwrite").saveAsTable(f"{DB}.manual_orders_silver")
    (spark.table(f"{DB}.manual_orders_silver").groupBy("customer_id").agg(F.sum("amount").alias("revenue"))
     .write.mode("overwrite").saveAsTable(f"{DB}.manual_revenue_by_customer"))
    return files

spark.sql(f"DROP TABLE IF EXISTS {DB}.manual_orders_bronze")
print("loaded files:", manual_update())
for t in ["manual_orders_bronze", "manual_orders_silver", "manual_revenue_by_customer"]:
    print(f"{t:<28} {spark.table(f'{DB}.{t}').count()} rows")

## 2. The declarative version

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the same pipeline as three SQL statements in a notebook in your workspace: a streaming table over the raw files, a streaming Silver table with expectations, and a materialized view for Gold.<br><br>
<b>Why it matters</b><br>There's no file tracking, no write mode and no ordering in this code. <code>STREAM read_files</code> processes each file once, <code>STREAM(ldp_orders_bronze)</code> reads Bronze incrementally, and the dependency graph comes from the table references.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The path of the pipeline source notebook.
</div>

In [0]:
import base64
from datetime import timedelta
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import pipelines, workspace

w = WorkspaceClient()
me = w.current_user.me().user_name
FOLDER = f"/Users/{me}/databricks-learning-workflows"
w.workspace.mkdirs(FOLDER)

PIPELINE_SQL = f"""
CREATE OR REFRESH STREAMING TABLE ldp_orders_bronze
COMMENT 'Raw orders, one row per CSV line'
AS SELECT *, _metadata.file_name AS source_file
FROM STREAM read_files('{RAW}/', format => 'csv', header => true,
                       schema => 'order_id INT, customer_id INT, amount DOUBLE, order_ts TIMESTAMP');

CREATE OR REFRESH STREAMING TABLE ldp_orders_silver (
  CONSTRAINT valid_amount EXPECT (amount > 0) ON VIOLATION DROP ROW,
  CONSTRAINT has_customer EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT recent_order EXPECT (order_ts >= '2024-01-01')
)
COMMENT 'Valid orders'
AS SELECT order_id, customer_id, amount, order_ts FROM STREAM(ldp_orders_bronze);

CREATE OR REFRESH MATERIALIZED VIEW ldp_revenue_by_customer
COMMENT 'Revenue per customer'
AS SELECT customer_id, SUM(amount) AS revenue, COUNT(*) AS orders FROM ldp_orders_silver GROUP BY customer_id;
"""
sql_path = f"{FOLDER}/ldp_orders_pipeline"
w.workspace.import_(path=sql_path, format=workspace.ImportFormat.SOURCE, language=workspace.Language.SQL,
                    content=base64.b64encode(PIPELINE_SQL.encode()).decode(), overwrite=True)
print(sql_path)

## 3. Create and run the pipeline

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a serverless, triggered pipeline in development mode that publishes to <code>workspace.workflows</code>, starts an update, and waits for it to finish.<br><br>
<b>Why it matters</b><br>Pipeline settings are separate from the code: the same SQL can be deployed to dev and prod with different catalogs. A triggered update processes everything available and stops, which is the usual choice for scheduled batch pipelines.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Free Edition supports serverless pipelines. The first update takes a few minutes. You can watch it live: <b>Jobs & Pipelines</b> → <code>dbl_04_orders_pipeline</code>. If creation fails, the cell prints the error. You can also create the pipeline in the UI with the same source notebook.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>COMPLETED</code>, then the three tables with 6, 4 and 3 rows.
</div>

In [0]:
for old in w.pipelines.list_pipelines(filter="name LIKE 'dbl_04_orders_pipeline'"):
    w.pipelines.delete(pipeline_id=old.pipeline_id)            # remove a pipeline left over from an earlier run

settings = dict(name="dbl_04_orders_pipeline", catalog=CATALOG, serverless=True, development=True, continuous=False,
                libraries=[pipelines.PipelineLibrary(notebook=pipelines.NotebookLibrary(path=sql_path))])
try:
    created = w.pipelines.create(schema="workflows", **settings)
except TypeError:                                              # older SDKs call the target schema "target"
    created = w.pipelines.create(target="workflows", **settings)
PIPELINE_ID = created.pipeline_id
print("pipeline:", f"{w.config.host}/pipelines/{PIPELINE_ID}")

def run_update(full_refresh=False):
    update_id = w.pipelines.start_update(pipeline_id=PIPELINE_ID, full_refresh=full_refresh).update_id
    while True:
        st = w.pipelines.get_update(pipeline_id=PIPELINE_ID, update_id=update_id).update.state.value
        if st in ("COMPLETED", "FAILED", "CANCELED"):
            return st
        time.sleep(15)

print("update:", run_update())
for t in ["ldp_orders_bronze", "ldp_orders_silver", "ldp_revenue_by_customer"]:
    print(f"{t:<26} {spark.table(f'{DB}.{t}').count()} rows")

## 4. New data: incremental processing

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes a second file with 3 more orders (one invalid), and runs another update.<br><br>
<b>Why it matters</b><br>The streaming tables read <b>only the new file</b>: Bronze grows by 3 rows instead of reloading everything. The materialized view is refreshed from Silver, incrementally when possible.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Bronze 9 rows, Silver 6, and updated revenue for the customers involved.
</div>

In [0]:
print(write_batch("batch_2", ["7,104,80.0,2024-11-03 10:00:00", "8,101,20.0,2024-11-03 11:00:00", "9,105,0.0,2024-11-03 12:00:00"]))

In [0]:
print("update:", run_update())
for t in ["ldp_orders_bronze", "ldp_orders_silver"]:
    print(f"{t:<26} {spark.table(f'{DB}.{t}').count()} rows")
display(spark.table(f"{DB}.ldp_revenue_by_customer").orderBy("customer_id"))

In [0]:
print("manual version, newly loaded files:", manual_update())
spark.table(f"{DB}.manual_revenue_by_customer").orderBy("customer_id").show()

## 5. Data quality from the event log

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Queries the pipeline's event log for the expectation results: how many records passed and failed each rule.<br><br>
<b>Why it matters</b><br>Every update records quality metrics automatically. The pipeline UI shows them per table, and the event log lets you build quality dashboards and alerts.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
For <code>ldp_orders_silver</code>: <code>valid_amount</code> failed 2 records in total (<code>-5.0</code> and <code>0.0</code>), <code>has_customer</code> failed 1, and <code>recent_order</code> failed 0 (it only warns).
</div>

In [0]:
display(spark.sql(f"""
    WITH dq AS (
      SELECT explode(from_json(details:flow_progress:data_quality:expectations,
                     'array<struct<name: string, dataset: string, passed_records: bigint, failed_records: bigint>>')) AS e
      FROM event_log('{PIPELINE_ID}')
      WHERE event_type = 'flow_progress' AND details:flow_progress:data_quality:expectations IS NOT NULL
    )
    SELECT e.dataset, e.name, SUM(e.passed_records) AS passed, SUM(e.failed_records) AS failed
    FROM dq GROUP BY ALL ORDER BY e.dataset, e.name
"""))

## 6. Schedule it, or clean up

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows how the pipeline is scheduled in production (a job with a pipeline task), then deletes the demo pipeline. Deleting a pipeline also deletes the tables it manages. Set <code>KEEP_PIPELINE = True</code> to keep everything for exploring.<br><br>
<b>Why it matters</b><br>Pipelines are usually triggered by a job, alongside other tasks (lesson 02), or by their own schedule.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>pipeline deleted</code> (or <code>pipeline kept</code>).
</div>

In [0]:
print("""jobs.Task(task_key="refresh_orders", pipeline_task=jobs.PipelineTask(pipeline_id=PIPELINE_ID, full_refresh=False))""")
KEEP_PIPELINE = False
if KEEP_PIPELINE:
    print("pipeline kept:", f"{w.config.host}/pipelines/{PIPELINE_ID}")
else:
    w.pipelines.delete(pipeline_id=PIPELINE_ID)
    print("pipeline deleted")

## Under the hood

```
update starts ─▶ parse SQL/Python ─▶ build the dataset graph:  ldp_orders_bronze ─▶ ldp_orders_silver ─▶ ldp_revenue_by_customer
  ├─ streaming tables: Structured Streaming with managed checkpoints (only new files / new rows)
  ├─ expectations: evaluated per row; metrics written to the event log
  ├─ materialized views: recomputed, incrementally when the query allows (Enzyme)
  └─ retries on failure (production mode), table maintenance handled automatically
full refresh ─▶ clears state and tables, reprocesses all source data
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> an update is the unit of execution, like a job run.

In [0]:
spark.table(f"{DB}.manual_orders_bronze").groupBy("source_file").count().orderBy("source_file").show()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: using a materialized view to ingest files</b><br>
Materialized views recompute from their sources. Ingest append-only sources with streaming tables (<code>STREAM read_files</code>).<br><br>
**⛔ Problem: <code>STREAM</code> reading a table that gets updates or deletes**<br>Streaming tables expect append-only sources. Use a materialized view, AUTO CDC, or the <code>skipChangeCommits</code> option.<br><br>
<b>⛔ Problem: changing a streaming table's logic and expecting old rows to change</b><br>Streaming tables only process new data. Run a <b>full refresh</b> of that table to reprocess history.<br><br>
<b>⛔ Problem: "table already exists" when creating a pipeline</b><br>A table with that name already exists in the target schema, managed by something else. Use another name or drop it.<br><br>
<b>⛔ Problem: development mode in production</b><br>Production mode retries failures and releases compute. Development mode is for iterating.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between a streaming table and a materialized view in Lakeflow Declarative Pipelines?</b><br>
A streaming table processes each input record once, incrementally, from an append-only source, which suits ingestion and append-style transformations. A materialized view holds the result of a query and is refreshed on each update (incrementally when possible), so it always reflects its sources, including updates and deletes. That suits aggregations and joins.<br><br>

<b>🎤 2. What are expectations, and what are the three violation actions?</b><br>
Data quality rules declared on a dataset. The default (warn) keeps the invalid records and records metrics. <code>DROP ROW</code> removes them. <code>FAIL UPDATE</code> stops the update. Metrics for all three appear in the event log and UI.<br><br>

<b>🎤 3. What does a declarative pipeline handle for you compared to hand-written Spark jobs?</b><br>
Dependency ordering from the queries, incremental processing and checkpoints, retries, compute management, data quality metrics, lineage and table maintenance.<br><br>

<b>🎤 4. What is a full refresh?</b><br>
An update that clears the pipeline's tables and streaming state and reprocesses all source data. It's used after logic changes that must apply to historical data.<br><br>

<b>🎤 5. Triggered vs continuous pipelines?</b><br>
Triggered pipelines process all available data and stop, usually on a schedule. Continuous pipelines keep running and process data as it arrives, for low latency at higher cost.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A pipeline must ingest new JSON files from a volume exactly once each. Which dataset type should be used?</b><br>
A. A materialized view<br>
B. A streaming table reading with <code>STREAM read_files(...)</code> (Auto Loader)<br>
C. A temporary view<br>
D. A standard view<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which expectation stops the pipeline update when a record violates it?</b><br>
A. <code>EXPECT (amount &gt; 0)</code><br>
B. <code>EXPECT (amount &gt; 0) ON VIOLATION DROP ROW</code><br>
C. <code>EXPECT (amount &gt; 0) ON VIOLATION FAIL UPDATE</code><br>
D. <code>CHECK (amount &gt; 0)</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b></details><br><br>

<b>Q3. Where are expectation pass/fail counts recorded?</b><br>
A. In <code>DESCRIBE HISTORY</code> of each table<br>
B. In the pipeline's event log (and shown in the pipeline UI)<br>
C. In <code>system.billing.usage</code><br>
D. Nowhere, unless you log them yourself<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Extend the manual version with a "Gold per day" table (revenue per order date), and notice how much code it takes</li><li>Write (as a string) the SQL for a declarative materialized view <code>ldp_revenue_by_day</code> doing the same</li><li>(Databricks) Recreate the pipeline with your extra view added to the SQL, run it, and query the new table</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Manual: one more write step to maintain, and to keep in the right order after Silver
(spark.table(f"{DB}.manual_orders_silver").groupBy(F.to_date("order_ts").alias("order_date"))
 .agg(F.sum("amount").alias("revenue")).write.mode("overwrite").saveAsTable(f"{DB}.manual_revenue_by_day"))
spark.table(f"{DB}.manual_revenue_by_day").orderBy("order_date").show()

# 2. Declarative: one statement; the dependency on Silver is inferred
BY_DAY_SQL = """
CREATE OR REFRESH MATERIALIZED VIEW ldp_revenue_by_day
AS SELECT to_date(order_ts) AS order_date, SUM(amount) AS revenue FROM ldp_orders_silver GROUP BY ALL;
"""
print(BY_DAY_SQL)

In [0]:
# 3. Recreate with the extra view
w.workspace.import_(path=sql_path, format=workspace.ImportFormat.SOURCE, language=workspace.Language.SQL,
                    content=base64.b64encode((PIPELINE_SQL + BY_DAY_SQL).encode()).decode(), overwrite=True)
try:
    created = w.pipelines.create(schema="workflows", **settings)
except TypeError:
    created = w.pipelines.create(target="workflows", **settings)
PIPELINE_ID = created.pipeline_id
print("update:", run_update())
display(spark.table(f"{DB}.ldp_revenue_by_day").orderBy("order_date"))
w.pipelines.delete(pipeline_id=PIPELINE_ID)

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Declare tables. The pipeline handles order, incremental state, retries and quality metrics</li><li>Streaming tables: append-only sources, each record once (Bronze/Silver). Materialized views: query results kept up to date (Silver/Gold)</li><li>Expectations: warn (default), <code>DROP ROW</code>, <code>FAIL UPDATE</code>. Metrics in the event log</li><li>Settings: catalog and schema, serverless, triggered vs continuous, development vs production</li><li>Full refresh reprocesses everything</li></ul>
</div>

| Need | SQL |
|---|---|
| Ingest files | `CREATE OR REFRESH STREAMING TABLE t AS SELECT * FROM STREAM read_files(...)` |
| Incremental from a table | `... FROM STREAM(upstream)` |
| Aggregate | `CREATE OR REFRESH MATERIALIZED VIEW mv AS SELECT ...` |
| Quality rule | `CONSTRAINT name EXPECT (cond) [ON VIOLATION DROP ROW / FAIL UPDATE]` |
| Quality metrics | `event_log('<pipeline_id>')`, `event_type = 'flow_progress'` |
| Python | `from pyspark import pipelines as dp` (`import dlt` in older code) |

## Git commit

```
git add 07_workflows/04_lakeflow_declarative_pipelines.ipynb
git commit -m "add 07_04 lakeflow declarative pipelines notebook"
```